# TP 0 — Understanding Decision Trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session2/tp_0_tree.ipynb)

## Objectives
1. Build intuition for how decision trees make predictions through sequential yes/no questions
2. Understand splitting criteria: Gini impurity, entropy, and information gain
3. Visualize how tree depth controls model complexity
4. Observe the overfitting/underfitting trade-off
5. Explore feature importance and regression trees

## Setup

Run the cell below to install and import the required packages.

In [ ]:
!pip install git+https://github.com/racousin/L2Math.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from l2math import plot_decision_boundary_2d

print("Setup complete!")

---
# Part 1: Intuition — A Game of Questions

A **decision tree** makes predictions by asking a sequence of **yes/no questions** about the input features. Each question splits the data into two groups, and the process repeats until a prediction is made.

**Conceptual example — Should I go outside?**

```
Is temperature > 30°C?
├── YES: Is humidity > 70%?
│   ├── YES → Stay inside (too hot and humid)
│   └── NO  → Go outside (hot but dry)
└── NO:  Is it raining?
    ├── YES → Stay inside
    └── NO  → Go outside
```

**Key vocabulary:**
- **Root node:** the first question (top of the tree)
- **Internal node:** an intermediate question
- **Leaf node:** a final prediction (no more questions)
- **Depth:** the number of questions from root to leaf

The central challenge is: **which question should we ask at each step?** We need a way to measure how "good" a split is. This is where splitting criteria come in.

---
# Part 2: Splitting Criteria — Classification

To decide the best split, we need to measure the **impurity** (or disorder) of a set of labels. A pure node contains samples from only one class.

## Gini Impurity

$$G = 1 - \sum_{k=1}^{K} p_k^2$$

where $p_k$ is the proportion of class $k$ in the node.

- $G = 0$: perfectly pure (all samples belong to one class)
- $G = 0.5$: maximum impurity for binary classification (50/50 split)

## Entropy

$$H = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

- $H = 0$: perfectly pure
- $H = 1$: maximum disorder for binary classification

Let's visualize both for the binary case ($K=2$), where the only free parameter is $p = p_1$ (and $p_2 = 1-p$).

In [ ]:
p = np.linspace(0.001, 0.999, 500)

# Gini impurity for binary case: G = 1 - p^2 - (1-p)^2 = 2p(1-p)
gini = 1 - p**2 - (1 - p)**2

# Entropy for binary case: H = -p*log2(p) - (1-p)*log2(1-p)
entropy = -p * np.log2(p) - (1 - p) * np.log2(1 - p)

plt.figure(figsize=(8, 5))
plt.plot(p, gini, 'b-', linewidth=2, label='Gini Impurity')
plt.plot(p, entropy, 'r--', linewidth=2, label='Entropy')
plt.xlabel('$p$ (proportion of class 1)', fontsize=12)
plt.ylabel('Impurity', fontsize=12)
plt.title('Gini Impurity vs Entropy (Binary Classification)', fontsize=13)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)
plt.show()

Both curves peak at $p = 0.5$ (maximum uncertainty) and equal zero at $p = 0$ or $p = 1$ (perfect purity). They are very similar in practice — sklearn uses Gini by default.

## Information Gain

A split is evaluated by how much it **reduces** impurity. The **information gain** of a split is:

$$IG = H(\text{parent}) - \sum_{i} \frac{n_i}{n} H(\text{child}_i)$$

where $n_i$ is the number of samples in child $i$ and $n$ is the total. The tree algorithm tries every possible feature and threshold, and picks the split with the **highest information gain**.

---
# Part 3: Worked Example — Computing Information Gain

Let's manually compute the information gain for different split points on a small dataset.

In [ ]:
np.random.seed(42)

# Small dataset: 10 samples, 1 feature, 2 classes
X_small = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
y_small = np.array([0, 0, 0, 0, 1, 1, 0, 1, 1, 1])  # labels

print("Dataset:")
print(f"  X = {X_small}")
print(f"  y = {y_small}")
print(f"  Class 0 count: {np.sum(y_small == 0)}, Class 1 count: {np.sum(y_small == 1)}")


def compute_entropy(y):
    """Compute entropy of a label array."""
    if len(y) == 0:
        return 0.0
    proportions = np.bincount(y) / len(y)
    proportions = proportions[proportions > 0]  # avoid log(0)
    return -np.sum(proportions * np.log2(proportions))


# Parent entropy
H_parent = compute_entropy(y_small)
print(f"\nParent entropy: H = {H_parent:.4f}")

# Try different split thresholds
thresholds = [2.5, 4.5, 6.5, 8.5]
print("\n" + "="*60)
print(f"{'Threshold':>10} | {'Left':>12} | {'Right':>12} | {'IG':>8}")
print("="*60)

best_ig = -1
best_threshold = None

for t in thresholds:
    left_mask = X_small <= t
    right_mask = X_small > t
    y_left = y_small[left_mask]
    y_right = y_small[right_mask]

    H_left = compute_entropy(y_left)
    H_right = compute_entropy(y_right)

    n = len(y_small)
    n_left = len(y_left)
    n_right = len(y_right)

    ig = H_parent - (n_left / n) * H_left - (n_right / n) * H_right

    if ig > best_ig:
        best_ig = ig
        best_threshold = t

    left_str = f"y={list(y_left)}"
    right_str = f"y={list(y_right)}"
    print(f"{t:>10.1f} | {left_str:>12} | {right_str:>12} | {ig:>8.4f}")

print("="*60)
print(f"\nBest split: X <= {best_threshold} (Information Gain = {best_ig:.4f})")

---
# Part 4: Splitting Criteria — Regression

For regression trees, the splitting criterion is based on **variance reduction** (equivalently, MSE reduction).

At each node, the prediction is the **mean** of the target values. The impurity is measured by the MSE within the node:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \bar{y})^2$$

The best split maximizes:

$$\Delta\text{MSE} = \text{MSE}(\text{parent}) - \frac{n_L}{n} \text{MSE}(\text{left}) - \frac{n_R}{n} \text{MSE}(\text{right})$$

In [ ]:
np.random.seed(42)

# Generate 1D data with a clear breakpoint
X_reg = np.sort(np.random.uniform(0, 10, 80))
y_reg = np.where(X_reg < 5, 2 + 0.5 * np.random.randn(80), 7 + 0.5 * np.random.randn(80))

# Find the best split point by brute force
split_candidates = np.linspace(1, 9, 200)
mse_reductions = []

parent_mse = np.mean((y_reg - np.mean(y_reg)) ** 2)

for s in split_candidates:
    left = y_reg[X_reg <= s]
    right = y_reg[X_reg > s]
    if len(left) == 0 or len(right) == 0:
        mse_reductions.append(0)
        continue
    mse_left = np.mean((left - np.mean(left)) ** 2)
    mse_right = np.mean((right - np.mean(right)) ** 2)
    reduction = parent_mse - (len(left) / len(y_reg)) * mse_left - (len(right) / len(y_reg)) * mse_right
    mse_reductions.append(reduction)

best_split = split_candidates[np.argmax(mse_reductions)]

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: data + split + region means
ax = axes[0]
left_mask = X_reg <= best_split
right_mask = X_reg > best_split
mean_left = np.mean(y_reg[left_mask])
mean_right = np.mean(y_reg[right_mask])

ax.scatter(X_reg, y_reg, c='steelblue', edgecolors='k', s=30, linewidths=0.5, zorder=3)
ax.axvline(x=best_split, color='red', linestyle='--', linewidth=2, label=f'Split at X = {best_split:.1f}')
ax.axhspan(mean_left - 0.3, mean_left + 0.3, xmin=0, xmax=best_split/10, alpha=0.2, color='orange')
ax.axhspan(mean_right - 0.3, mean_right + 0.3, xmin=best_split/10, xmax=1, alpha=0.2, color='green')
ax.hlines(mean_left, X_reg.min(), best_split, colors='orange', linewidth=2, label=f'Left mean = {mean_left:.2f}')
ax.hlines(mean_right, best_split, X_reg.max(), colors='green', linewidth=2, label=f'Right mean = {mean_right:.2f}')
ax.set_xlabel('X', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_title('Regression Split: Data and Predictions', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right panel: MSE reduction vs split point
ax = axes[1]
ax.plot(split_candidates, mse_reductions, 'b-', linewidth=2)
ax.axvline(x=best_split, color='red', linestyle='--', linewidth=2, label=f'Best split = {best_split:.1f}')
ax.set_xlabel('Split point', fontsize=11)
ax.set_ylabel('MSE Reduction', fontsize=11)
ax.set_title('MSE Reduction vs Split Point', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Optimal split point: X = {best_split:.2f}")
print(f"MSE reduction: {max(mse_reductions):.4f}")

---
# Part 5: Building a Tree Step by Step

Let's watch a decision tree grow on a 2D classification problem. We use `make_moons` which creates two interleaving half-circles — a classic nonlinear problem.

We increase `max_depth` from 1 to unlimited and observe how the decision boundary becomes more complex.

In [ ]:
np.random.seed(42)

# Generate 2D classification data
X_moons, y_moons = make_moons(n_samples=200, noise=0.3, random_state=42)

# Train trees at different depths
depths = [1, 2, 3, None]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, depth in enumerate(depths):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_moons, y_moons)
    acc = accuracy_score(y_moons, tree.predict(X_moons))
    depth_label = str(depth) if depth is not None else "None (unlimited)"
    plot_decision_boundary_2d(
        tree, X_moons, y_moons,
        title=f"max_depth={depth_label}  (train acc={acc:.2f})",
        ax=axes[i]
    )

plt.suptitle('Decision Tree — Effect of Depth on Decision Boundary', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- `max_depth=1`: Only one split — the boundary is a single horizontal or vertical line. Too simple.
- `max_depth=2`: Two levels of splits — starts to capture the moon shape.
- `max_depth=3`: More splits — a better approximation of the true boundary.
- `max_depth=None`: The tree grows until every leaf is pure. Very complex boundary — likely overfitting to noise.

---
# Part 6: Visualizing Tree Structure

Let's look at the internal structure of a depth-3 tree. Each node shows:
- The **split condition** (e.g., $X[0] \leq 0.5$)
- The **Gini impurity**
- The **number of samples**
- The **class distribution**

In [ ]:
tree_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_depth3.fit(X_moons, y_moons)

plt.figure(figsize=(20, 10))
plot_tree(
    tree_depth3,
    filled=True,
    rounded=True,
    feature_names=['Feature 1', 'Feature 2'],
    class_names=['Class 0', 'Class 1'],
    fontsize=11
)
plt.title('Decision Tree Structure (max_depth=3)', fontsize=15)
plt.tight_layout()
plt.show()

print(f"Number of leaves: {tree_depth3.get_n_leaves()}")
print(f"Tree depth: {tree_depth3.get_depth()}")

---
# Part 7: Overfitting with Depth

A deeper tree can perfectly fit the training data, but it may perform poorly on unseen data. Let's measure **train and test accuracy** as a function of `max_depth`.

In [ ]:
np.random.seed(42)

# Generate data and split
X_moons, y_moons = make_moons(n_samples=300, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42
)

# Evaluate accuracy for different depths
depth_range = range(1, 21)
train_accs = []
test_accs = []

for depth in depth_range:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, tree.predict(X_train)))
    test_accs.append(accuracy_score(y_test, tree.predict(X_test)))

# Plot
plt.figure(figsize=(9, 5))
plt.plot(depth_range, train_accs, 'o-', color='blue', linewidth=2, label='Train accuracy')
plt.plot(depth_range, test_accs, 's-', color='red', linewidth=2, label='Test accuracy')
plt.xlabel('max_depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Train vs Test Accuracy as a Function of Tree Depth', fontsize=13)
plt.legend(fontsize=12)
plt.xticks(depth_range)
plt.grid(True, alpha=0.3)
plt.ylim(0.7, 1.02)
plt.show()

best_depth = depth_range[np.argmax(test_accs)]
print(f"Best test accuracy: {max(test_accs):.4f} at max_depth={best_depth}")
print(f"Train accuracy at that depth: {train_accs[best_depth - 1]:.4f}")

**Key insight:**
- **Training accuracy** increases monotonically with depth (the tree memorizes the training data).
- **Test accuracy** first increases, then plateaus or decreases — this is **overfitting**.
- The optimal `max_depth` balances complexity and generalization.

---
# Part 8: Feature Importance

Decision trees provide a natural measure of **feature importance**: features used for splitting near the root (where they affect the most samples) are considered more important.

In sklearn, `tree.feature_importances_` gives the normalized total reduction of the splitting criterion brought by each feature.

In [ ]:
np.random.seed(42)

# Generate data with 10 features, only 3 are informative
X_imp, y_imp = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=3,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)

# Train a deep tree
tree_imp = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_imp.fit(X_imp, y_imp)

# Extract feature importances
importances = tree_imp.feature_importances_
feature_names = [f'Feature {i}' for i in range(10)]

# Sort by importance
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
colors = ['#e74c3c' if i < 3 else '#3498db' for i in sorted_idx]
plt.bar(range(10), importances[sorted_idx], color=colors, edgecolor='k', linewidth=0.5)
plt.xticks(range(10), [feature_names[i] for i in sorted_idx], rotation=45, ha='right')
plt.ylabel('Feature Importance', fontsize=12)
plt.title('Feature Importances (red = truly informative features)', fontsize=13)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Feature importances (sorted):")
for i in sorted_idx:
    informative_tag = " (informative)" if i < 3 else ""
    print(f"  {feature_names[i]}: {importances[i]:.4f}{informative_tag}")

**Observation:** The tree correctly assigns high importance to the 3 informative features and low or zero importance to the 7 noise features. This makes decision trees useful for understanding which features matter.

---
# Part 9: Regression Trees

Decision trees can also solve **regression** problems. Instead of predicting a class, each leaf predicts the **mean** of the target values in that region. The result is a **piecewise constant** approximation.

Let's fit regression trees of different depths to a noisy sine wave.

In [ ]:
np.random.seed(42)

# Generate noisy sine data
X_sine = np.sort(np.random.uniform(0, 2 * np.pi, 200))
y_sine = np.sin(X_sine) + 0.3 * np.random.randn(200)

# Fine grid for predictions
X_plot = np.linspace(0, 2 * np.pi, 1000).reshape(-1, 1)

# Fit trees with different depths
depths_reg = [1, 3, 10]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for i, depth in enumerate(depths_reg):
    ax = axes[i]
    reg_tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    reg_tree.fit(X_sine.reshape(-1, 1), y_sine)
    y_pred_plot = reg_tree.predict(X_plot)

    ax.scatter(X_sine, y_sine, c='steelblue', s=15, alpha=0.5, edgecolors='none', label='Data')
    ax.plot(X_plot, y_pred_plot, 'r-', linewidth=2, label='Tree prediction')
    ax.plot(X_plot, np.sin(X_plot), 'k--', linewidth=1, alpha=0.5, label='True function')
    ax.set_xlabel('X', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_title(f'max_depth={depth} ({reg_tree.get_n_leaves()} leaves)', fontsize=13)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-2, 2)

plt.suptitle('Regression Trees — Piecewise Constant Approximation', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Key insight:**
- `max_depth=1`: Only one split — very rough approximation (underfitting).
- `max_depth=3`: 8 regions — captures the general sine shape.
- `max_depth=10`: Many small regions — closely follows the data, including noise (overfitting).

Notice that regression tree predictions are always **piecewise constant** (staircase-like), because each leaf outputs a single value.

---
# Summary

## How Decision Trees Work
- A decision tree recursively splits the feature space using axis-aligned thresholds.
- **Classification:** splits minimize Gini impurity or entropy; leaves predict the majority class.
- **Regression:** splits minimize MSE; leaves predict the mean of target values.

## Pros
- Simple to understand and interpret (white-box model)
- No feature scaling needed
- Handles both numerical and categorical features
- Built-in feature importance

## Cons
- **Prone to overfitting** (especially deep trees)
- **High variance:** small changes in data can produce very different trees
- Axis-aligned splits struggle with diagonal decision boundaries
- Predictions are piecewise constant (for regression)

## Key Hyperparameters
| Hyperparameter | Effect |
|---|---|
| `max_depth` | Maximum depth of the tree. Lower = simpler model. |
| `min_samples_split` | Minimum samples required to split a node. Higher = simpler model. |
| `min_samples_leaf` | Minimum samples in a leaf. Higher = simpler model. |
| `criterion` | Splitting criterion: `'gini'` or `'entropy'` (classification), `'squared_error'` (regression). |